**The purpose of this notebook:** apply the clean function and attend the specific needs of the dataset to The final cleaned dataset to be used for further analysis and modeling.

In [1]:
import pandas as pd
import yaml
import openpyxl

---

In [2]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config").is_dir())
with open(ROOT / "config" / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg = {k: str(ROOT / v) for k, v in cfg.items()}  # relative paths resolve against the project root

In [3]:
P_A1_24 = pd.read_excel(cfg["P_A1_24_path"], engine="openpyxl")

---

In [4]:
import sys
sys.path.append("../..")
from src.functions import clean_plant

In [5]:
P_A1_24_clean = clean_plant(P_A1_24)
P_A1_24_clean.head()

Rows with all NaN values: 6794
Rows with 'Dosif. m3' in first column: 6794


,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5R",Cemento\n 4\n CEM III/A 42,Cemento\n 6\n CEM IV/A(V),Cemento\n 7\n CEM IV/A(V),Arido\n 1\n ARENA 0-4,Arido\n 2\n ARIDO 8-14,Arido\n 3\n ARIDO 12-20,Agua\n 1\n AGUA,Aditivo\n 1\n COMPLAST MR-,Aditivo\n 3\n DYNAMON SX-3
0,2024-12-27,HL-150/B/20/1,4.0,602.0,0.0,0.0,0.0,4724.0,1590.0,1990.0,523.0,3.9,0.0
1,2024-12-27,HL-150/B/20/1,7.0,1049.0,0.0,0.0,0.0,8440.0,2794.0,3493.0,911.0,6.9,0.0
2,2024-12-27,HM-20/B/20/I/1,10.0,2022.0,0.0,0.0,0.0,11971.0,3492.0,5274.0,1288.0,16.2,0.0
3,2024-12-27,HA-30/B/20/XC2/1,6.0,1751.0,0.0,0.0,0.0,6651.0,1938.0,3391.0,958.0,11.4,8.9
4,2024-12-27,HL-150/B/20/1,5.0,749.0,0.0,0.0,0.0,5912.0,1987.0,2504.0,651.0,4.9,0.0


---

In [6]:
# How many non-zero values each column has and what percentage of all rows that represents.
non_zero_counts = (P_A1_24_clean != 0.0).sum()
non_zero_pct = non_zero_counts / len(P_A1_24_clean) * 100

pd.DataFrame({
    "non_zero_count": non_zero_counts,
    "non_zero_pct": non_zero_pct.round(2)
})

,non_zero_count,non_zero_pct
\n \n Fecha,6794,100.00
\n \n Fórmula,6794,100.00
\n \n Volumen,6792,99.97
"Cemento\n 1\n CEM I 42,5R",707,10.41
Cemento\n 4\n CEM III/A 42,119,1.75
Cemento\n 6\n CEM IV/A(V),5821,85.68
Cemento\n 7\n CEM IV/A(V),85,1.25
Arido\n 1\n ARENA 0-4,6736,99.15
Arido\n 2\n ARIDO 8-14,6615,97.37
Arido\n 3\n ARIDO 12-20,5648,83.13


In [8]:
P_A1_24_clean.shape

(6794, 13)

In [9]:
P_A1_24_clean.columns = ["date", "product", "volume", "cement_42_5r", "cement_3a_42", "cement_4av","cement_4av_1", "arid_04", "arid_8_14", "arid_12_20","water","ad_complast_mr","ad_dynamon_sx"]

In [10]:
P_A1_24_clean.columns.tolist()

['date',
 'product',
 'volume',
 'cement_42_5r',
 'cement_3a_42',
 'cement_4av',
 'cement_4av_1',
 'arid_04',
 'arid_8_14',
 'arid_12_20',
 'water',
 'ad_complast_mr',
 'ad_dynamon_sx']

In [11]:
# Frequency of rows where any of two or more of the following products; 
# "cement_42_5r", "cement_3a_42", "cement_4av","cement_4av_1" are !=0 in the same row.
cement_columns = ["cement_42_5r", "cement_3a_42", "cement_4av", "cement_4av_1"]

# Count non-zero values per row for these columns.
non_zero_cement_count = (P_A1_24_clean[cement_columns] != 0).sum(axis=1)

# Find rows where 2 or more cement products are non-zero.
condition = non_zero_cement_count >= 2
frequency = condition.sum()

print(f"Frequency of rows where 2 or more cement products are non-zero: {frequency}")
print(f"Percentage: {(frequency / len(P_A1_24_clean) * 100):.2f}%")

Frequency of rows where 2 or more cement products are non-zero: 0
Percentage: 0.00%


In [12]:
# Which means all loads used a unique cement type. (When a concrete product need cement).
# We can merge "cement_4av", "cement_4av_1" into one column "cement_4av" by summing them up.
P_A1_24_clean["cement_4av"] = P_A1_24_clean["cement_4av"] + P_A1_24_clean["cement_4av_1"]
P_A1_24_clean.drop(columns=["cement_4av_1"], inplace=True)

In [13]:
## we add a new where all values will be P_A1 because is the future merging we need to identify from which plant the load is come from. 
# Inserting a new column and value. 
# P_A1_24_clean["plant"] = "P_A1"
# Alternative way to inserting a column with values and the position too. 
P_A1_24_clean.insert(1, "plant", "P_A1")

In [14]:
P_A1_24_clean.columns.tolist()

['date',
 'plant',
 'product',
 'volume',
 'cement_42_5r',
 'cement_3a_42',
 'cement_4av',
 'arid_04',
 'arid_8_14',
 'arid_12_20',
 'water',
 'ad_complast_mr',
 'ad_dynamon_sx']

---

In [15]:
P_A1_24_clean.to_csv(cfg["P_A1_24_ready_csv_path"], index=False, encoding="utf-8-sig")